## Plot the trajectory and eigenvectors for selected events.

### 1. Import all packages and calculate standard BS and MP distance.

In [ ]:
import pandas as pd
import datetime as dt
import numpy as np 
import glob
import matplotlib.pyplot as plt 
import re

from matplotlib.ticker import AutoMinorLocator

from JunoPy.spice import InitSpiceKernels, get_juno_position
from JunoPy.mag import MagData, mva


def calc_coeff_bs(p):
    A = -1.107 + 1.591 * p**(-0.25)
    B = -0.566 - 0.812 * p**(-0.25)
    C =  0.048 - 0.059 * p**(-0.25)
    D =  0.077 - 0.038 * p
    E = -0.874 - 0.299 * p
    F = -0.055 + 0.124 * p
    return A, B, C, D, E, F
    
def calc_coeff_mp(p):
    A = -0.134 + 0.488 * p**(-0.25)
    B = -0.581 - 0.225 * p**(-0.25)
    C = -0.186 - 0.016 * p**(-0.25)
    D = -0.014 + 0.096 * p
    E = -0.814 - 0.811 * p
    F = -0.050 + 0.168 * p
    return A, B, C, D, E, F

def calc_equatorial(p, N, ctype='MP'):
    
    if 'MP' in ctype:
        A, B, C, D, E, F = calc_coeff_mp(p)
    else:
        A, B, C, D, E, F = calc_coeff_bs(p)
        
    phi = np.linspace(-150, 150, N) * np.pi / 180
    
    a = C * np.cos(phi)**2 + E * np.sin(phi)**2 + F * np.sin(phi) * np.cos(phi)
    b = B * np.cos(phi) + D * np.sin(phi)
    c = A
    
    root1 = (-b + np.sqrt(b**2 - 4 * a * c)) / (2 * a)
    root2 = (-b - np.sqrt(b**2 - 4 * a * c)) / (2 * a)
    
    r = np.zeros(N)
    for i in range(N):
        if root1[i] > 0:
            r[i] = root1[i]
        else:
            r[i] = root2[i]
        
    x = r * 120 * np.cos(phi)
    y = r * 120 * np.sin(phi)    
    
    return x, y

def calc_terminator(p, N, ctype='MP'):
    
    if 'MP' in ctype:
        A, B, C, D, E, F = calc_coeff_mp(p)
    else:
        A, B, C, D, E, F = calc_coeff_bs(p)
        
    theta = np.linspace(0, 180, N) * np.pi / 180
    
    a = E * np.sin(theta)**2 - np.cos(theta)**2 
    b = -D * np.sin(theta)
    c = A
    
    root1 = (-b + np.sqrt(b**2 - 4 * a * c)) / (2 * a)
    root2 = (-b - np.sqrt(b**2 - 4 * a * c)) / (2 * a)
    
    r = np.zeros(N)
    for i in range(N):
        if root1[i] > 0:
            r[i] = root1[i]
        else:
            r[i] = root2[i]
    
    y = -r * 120 * np.sin(theta)
    z = r * 120 * np.cos(theta)
    return y, z


x_MP, y_MP = calc_equatorial(0.383, 101, ctype='MP')
x_BS, y_BS = calc_equatorial(0.382, 101, ctype='BS')
y_MP_term, z_MP_term = calc_terminator(0.383, 101, ctype='MP')
y_BS_term, z_BS_term = calc_terminator(0.382, 101, ctype='BS')

### 2. Get the Juno trajectory for the entire duration.

In [ ]:
start = dt.datetime(2016, 6, 20, 17, 30)
end   = dt.datetime(2019, 5, 31, 16)

time_array = pd.date_range(
        start=start, end=end, periods=10000).to_pydatetime().tolist()

InitSpiceKernels()
position, _ = get_juno_position(time_array)

### 3. Read the CSV file and get Juno position for the flux rope events

In [ ]:
def read_csv_file():
    starttimes = []
    endtimes = []
    durations = []
    PNevents = []
    NPevents = []
    with open('mva_3events.csv', 'r') as f:
        f.readline()
        
        for i, line in enumerate(f):    

            starttimes.append(dt.datetime.strptime(line.split(',')[0], '%Y-%jT%H:%M:%S'))
            endtimes.append(dt.datetime.strptime(line.split(',')[1], '%Y-%jT%H:%M:%S'))
            durations.append(float(line.split(',')[2]))
            
            if 'PN' in line.split(',')[3]:
                PNevents.append(i)
            else:
                NPevents.append(i)
            
    return starttimes, endtimes, durations, PNevents, NPevents
            
starttimes, endtimes, durations, PNevents, NPevents = read_csv_file()
InitSpiceKernels()
position_mva, _ = get_juno_position(starttimes)



### 4. Perform MVA analysis and get the direction of min/mid/max variance

In [ ]:
def read_and_crop(magfile, start, end):
    """Read and crop magnetometer data."""
    magdata = MagData(filename=magfile)
    start_cut = start - dt.timedelta(minutes=10)
    end_cut   = end   + dt.timedelta(minutes=10)
    magdata.df = magdata.df.loc[start_cut:end_cut]

    magdata_mva = MagData(filename=magfile)
    magdata_mva.df = magdata_mva.df.loc[start:end]
    magdata_mva = mva(magdata_mva)

    return magdata, magdata_mva

x1 = []
x2 = []
x3 = []
for start, end in zip(starttimes, endtimes):
    
    magfile = glob.glob('data/mag/*{:s}ss*.hdf5'.format(start.strftime('%Y%j')))[0]
    magdata, magdata_mva = read_and_crop(magfile, start, end)
    x1.append(magdata_mva.x1)
    x2.append(magdata_mva.x2)
    x3.append(magdata_mva.x3)
    
x1 = np.asarray(x1)
x2 = np.asarray(x2)
x3 = np.asarray(x3)

for i in range(len(x3)):
    if x3[i, 2] < 0:
        x1[i, :] = x1[i, :] * -1
        x2[i, :] = x2[i, :] * -1
        x3[i, :] = x3[i, :] * -1

### 5. Read in the streamtraces from the MHD model.

In [ ]:
def read_streamtraces_file(filename):
    
    nZones = 0
    streamtraces = []
    
    with open(filename, 'r') as f:
        
        for line in f:
            
            if 'ZONE' in line:
                
                nZones += 1
                
                f.readline()
                npntsline = f.readline()
                nPoints = int(re.search('I=(\d+)', npntsline)[1])
                
                f.readline()
                f.readline()
                coords = np.zeros((nPoints, 3))
                for i in range(nPoints):
                    coords[i, :] = [float(x) for x in f.readline().split()]
                
                streamtraces.append(coords)
    
    
    return streamtraces

streamtraces = read_streamtraces_file('streamtraces-reduced-2.dat')
streamtraces3d = read_streamtraces_file('streamtraces-3d-4.dat')

### 6. Make the plot

In [ ]:
from matplotlib import transforms
def rainbow_text(x, y, strings, colors, orientation='horizontal',
                 ax=None, **kwargs):
    """
    Take a list of *strings* and *colors* and place them next to each
    other, with text strings[i] being shown in colors[i].

    Parameters
    ----------
    x, y : float
        Text position in data coordinates.
    strings : list of str
        The strings to draw.
    colors : list of color
        The colors to use.
    orientation : {'horizontal', 'vertical'}
    ax : Axes, optional
        The Axes to draw into. If None, the current axes will be used.
    **kwargs
        All other keyword arguments are passed to plt.text(), so you can
        set the font size, family, etc.
    """
    if ax is None:
        ax = plt.gca()
    t = ax.transData
    canvas = ax.figure.canvas

    assert orientation in ['horizontal', 'vertical']
    if orientation == 'vertical':
        kwargs.update(rotation=90, verticalalignment='bottom')

    for s, c in zip(strings, colors):
        # text = ax.text(x, y, s + " ", color=c, transform=t, **kwargs)
        text = ax.text(x, y, s + ' ', color=c, transform=t, **kwargs)

        # Need to draw to update the text position.
        text.draw(canvas.get_renderer())
        ex = text.get_window_extent()
        if orientation == 'horizontal':
            t = transforms.offset_copy(
                text.get_transform(), x=15, units='points', fig=ax.figure) # x=ex.width, units='dots')

        else:
            t = transforms.offset_copy(
                text.get_transform(), y=ex.height, units='dots')

In [ ]:
# fig = plt.figure(figsize=(8, 4), sharex=True, constrained_layout=False, dpi=600)
gs_kw = dict(height_ratios=[130, 65], hspace=0.1, width_ratios=[1, 1], wspace=0)
fig, ax = plt.subplots(2, 2, dpi=600, gridspec_kw=gs_kw, figsize=(12, 8))

ax1 = ax[0, 0]
ax2 = ax[1, 0]
ax3 = ax[1, 1] 
ax[0, 1].set_axis_off()

ax1.plot(position[:, 0], position[:, 1], color='0.5', zorder=1)
ax2.plot(position[:, 0], position[:, 2], color='0.5', zorder=1)
ax3.plot(position[:, 1], position[:, 2], color='0.5', zorder=1)

# ax1.plot(position_masked[:, 0], position_masked[:, 1], color=[0.3, 0.3, 0.3], zorder=1)
# ax2.plot(position_masked[:, 0], position_masked[:, 2], color=[0.3, 0.3, 0.3], zorder=1)
    
ax1.plot(x_BS, y_BS, color='k', zorder=0)
ax1.plot(x_MP, y_MP, color='k', ls='dashed', zorder=0)

ax3.plot(y_BS_term, z_BS_term, color='k', zorder=0)
ax3.plot(y_MP_term, z_MP_term, color='k', ls='dashed', zorder=0)

for stream in streamtraces:
    ax2.plot(stream[:, 0], stream[:, 2], color='blue', alpha=0.7, zorder=0, lw=1.0)
    
# for stream in streamtraces3d:
#     ax1.plot(stream[:, 0], stream[:, 1], color='blue', alpha=0.7, zorder=0, lw=1.0)

ax1.xaxis.set_minor_locator(AutoMinorLocator())
ax1.yaxis.set_minor_locator(AutoMinorLocator())
ax2.xaxis.set_minor_locator(AutoMinorLocator())
ax2.yaxis.set_minor_locator(AutoMinorLocator())
ax3.xaxis.set_minor_locator(AutoMinorLocator())
ax3.yaxis.set_minor_locator(AutoMinorLocator())
ax1.grid('on', which='minor', lw=0.5, ls='dotted')
ax1.grid(color='k', ls='dashed', lw=0.25)
ax2.grid('on', which='minor', lw=0.5, ls='dotted')
ax2.grid(color='k', ls='dashed', lw=0.25)
ax3.grid('on', which='minor', lw=0.5, ls='dotted')
ax3.grid(color='k', ls='dashed', lw=0.25)

ax1.set_xlabel('X [R]', weight='bold')
ax1.set_ylabel('Y [R]', weight='bold')
ax2.set_xlabel('X [R]', weight='bold')
ax2.set_ylabel('Z [R]', weight='bold')
ax3.set_xlabel('Y [R]', weight='bold')
ax3.set_ylabel('Z [R]', weight='bold', labelpad=-5)

# ax1.set_xlim([-105, 40])
ax1.set_xlim([-105, 40])
ax1.set_ylim([-120, 10])

ax2.set_xlim([-105, 40])
ax2.set_ylim([-45, 20])

ax3.set_xlim([-120, 10])
ax3.set_ylim([-45, 20])

# ax1.set_adjustable("datalim", share=True)

ax3.get_shared_y_axes().join(ax2, ax3)
# ax3.set_yticklabels([])

ax1.set_aspect('equal', share=True)
ax2.set_aspect('equal', share=True)
ax3.set_aspect('equal', share=True)

sc1 = ax1.scatter(position_mva[:, 0], position_mva[:, 1], c='0.0', zorder=2, marker='o', cmap='jet')
sc2 = ax2.scatter(position_mva[:, 0], position_mva[:, 2], c='0.0', zorder=2, marker='o', cmap='jet')
sc3 = ax3.scatter(position_mva[:, 1], position_mva[:, 2], c='0.0', zorder=2, marker='o', cmap='jet')

ax1.quiver(position_mva[:, 0], position_mva[:, 1], x1[:, 0], x1[:, 1], color='g', scale=10, zorder=3)
ax1.quiver(position_mva[:, 0], position_mva[:, 1], x2[:, 0], x2[:, 1], color='b', scale=10, zorder=3)
ax1.quiver(position_mva[:, 0], position_mva[:, 1], x3[:, 0], x3[:, 1], color='r', scale=10, zorder=3)

ax2.quiver(position_mva[:, 0], position_mva[:, 2], x1[:, 0], x1[:, 2], color='g', scale=10, zorder=3)
ax2.quiver(position_mva[:, 0], position_mva[:, 2], x2[:, 0], x2[:, 2], color='b', scale=10, zorder=3)
ax2.quiver(position_mva[:, 0], position_mva[:, 2], x3[:, 0], x3[:, 2], color='r', scale=10, zorder=3)

ax3.quiver(position_mva[:, 1], position_mva[:, 2], x1[:, 1], x1[:, 2], color='g', scale=10, zorder=3)
ax3.quiver(position_mva[:, 1], position_mva[:, 2], x2[:, 1], x2[:, 2], color='b', scale=10, zorder=3)
ax3.quiver(position_mva[:, 1], position_mva[:, 2], x3[:, 1], x3[:, 2], color='r', scale=10, zorder=3)

# bbox2 = ax2.get_position()
# bbox1 = ax1.get_position()
# cax = fig.add_axes([bbox2.x1, bbox2.y0, 0.016, bbox1.y1- bbox2.y0])


ax1.arrow(20, -115, 10, 0, color='k', head_width=3)
ax1.text(20, -110, 'To Sun')
ax1.text(30, -60, 'MP')
ax1.text(30, -90, 'BS')

rainbow_text(-100, 2, ['L', 'M', 'N'], ['g', 'b', 'r'], ax=ax1, weight='bold')
rainbow_text(-100, 2, ['L', 'M', 'N'], ['g', 'b', 'r'], ax=ax2, weight='bold')

ax2.text(-60, -40, 'Juno Trajectory', color='0.5')
ax2.text(-5, -40, 'B field lines', color='blue')

ax1.text(-35, -65, '#1', color='k', zorder=3, weight='bold')
ax1.text(-55, -78, '#2', color='k', zorder=3, weight='bold')
# ax1.text(-33, -15, '#3', color='k', zorder=3, weight='bold')
ax1.text(2.0, 2.0, 'Jupiter', color='k', zorder=3, weight='bold')

ax1.text(30, 0, 'a)', color='k', zorder=3, weight='bold')
ax2.text(30, 10, 'b)', color='k', zorder=3, weight='bold')
ax3.text(0, 10, 'c)', color='k', zorder=3, weight='bold')

ax1.add_artist(plt.Circle((0, 0), 1.0, color='k', zorder=0))
ax2.add_artist(plt.Circle((0, 0), 1.0, color='k', zorder=0))
ax3.add_artist(plt.Circle((0, 0), 1.0, color='k', zorder=0))

# cbar2 = fig.colorbar(sc2, cax=cax);
# cbar2.set_label('Peak-to-peak Duration (s)')
# plt.subplots_adjust(wspace=0.0)
plt.tight_layout()
fig.savefig('out_images/TrajectoryLocationofEvent_quiver_3events.eps', format='eps')

## Make a 3D projection plot

In [ ]:
# def set_axes_equal(ax):
#     '''Make axes of 3D plot have equal scale so that spheres appear as spheres,
#     cubes as cubes, etc..  This is one possible solution to Matplotlib's
#     ax.set_aspect('equal') and ax.axis('equal') not working for 3D.

#     Input
#       ax: a matplotlib axis, e.g., as output from plt.gca().
#     '''

#     x_limits = ax.get_xlim3d()
#     y_limits = ax.get_ylim3d()
#     z_limits = ax.get_zlim3d()

#     x_range = abs(x_limits[1] - x_limits[0])
#     x_middle = np.mean(x_limits)
#     y_range = abs(y_limits[1] - y_limits[0])
#     y_middle = np.mean(y_limits)
#     z_range = abs(z_limits[1] - z_limits[0])
#     z_middle = np.mean(z_limits)

#     # The plot bounding box is a sphere in the sense of the infinity
#     # norm, hence I call half the max range the plot radius.
#     plot_radius = 0.3*max([x_range, y_range, z_range])

#     ax.set_xlim3d([x_middle - plot_radius, x_middle + plot_radius])
#     ax.set_ylim3d([y_middle - plot_radius, y_middle + plot_radius])
#     ax.set_zlim3d([z_middle - plot_radius, z_middle + plot_radius])
    
#     return ax
    
# fig = plt.figure(dpi=600)

# ax = fig.add_subplot(111, projection='3d', aspect='equal')

# ax.plot(position[:, 0], position[:, 1], zdir='z', zs=-40, color='0.6', zorder=1, alpha=0.5, lw=0.5)
# ax.plot(position[:, 0], position[:, 2], zdir='y', zs=10, color='0.6', zorder=1, alpha=0.5, lw=0.5)
# ax.plot(position[:,
1], position[:, 2], zdir='x', zs=-100, color='0.6', zorder=1, alpha=0.5, lw=0.5)

# ax.plot(position[:, 0], position[:, 1], position[:, 2], color='k', zorder=1, lw=1.0)
# ax = set_axes_equal(ax)

